In [74]:
%pip uninstall -y ml-dtypes jax jaxlib
%pip install "ml_dtypes>=0.3.1,<0.4"
# (po želji) osiguraj TF za Apple Silicon:
%pip install -U tensorflow-macos tensorflow-metal
!pip install -U tensorflow-macos tensorflow-metal
!pip install -q tensorflow-macos tensorflow-metal || pip install -q tensorflow


Found existing installation: ml_dtypes 0.5.3
Uninstalling ml_dtypes-0.5.3:
  Successfully uninstalled ml_dtypes-0.5.3
Found existing installation: jax 0.7.1
Uninstalling jax-0.7.1:
  Successfully uninstalled jax-0.7.1
Found existing installation: jaxlib 0.7.1
Uninstalling jaxlib-0.7.1:
  Successfully uninstalled jaxlib-0.7.1
Note: you may need to restart the kernel to use updated packages.
  Using cached ml_dtypes-0.3.2-cp311-cp311-macosx_10_9_universal2.whl.metadata (20 kB)
Using cached ml_dtypes-0.3.2-cp311-cp311-macosx_10_9_universal2.whl (389 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip 

In [75]:
!pip install mediapipe opencv-python


  Using cached jax-0.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.7.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (1.3 kB)
  Using cached ml_dtypes-0.5.3-cp311-cp311-macosx_10_9_universal2.whl.metadata (8.9 kB)
Using cached jax-0.7.1-py3-none-any.whl (2.8 MB)
Using cached jaxlib-0.7.1-cp311-cp311-macosx_11_0_arm64.whl (57.7 MB)
Using cached ml_dtypes-0.5.3-cp311-cp311-macosx_10_9_universal2.whl (667 kB)
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [jax]2/3 [jax]ib]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.2 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To upd

In [76]:
import mediapipe as mp
import cv2
import numpy as np
import uuid
import os


In [77]:
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands


In [78]:
pip install -q scikit-learn joblib matplotlib seaborn



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [87]:
# Prikupljanje podataka: spremi 21*3 = 63 vrijednosti po ruci + handedness
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

SAMPLES_PER_CLASS = 80  # promijeni po potrebi
SAVE_PAUSE_SEC = 0.1

print("Oznake:", GESTURE_LABELS)
print("Pritisni broj 0-{} za labelu, 's' za spremanje uzorka, 'q' za izlaz".format(len(GESTURE_LABELS)-1))

current_label_idx = None

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera nije dostupna")

with mp_hands.Hands(static_image_mode=False,
                    max_num_hands=1,
                    min_detection_confidence=0.7,
                    min_tracking_confidence=0.5) as hands:
    saved_counts = {label: 0 for label in GESTURE_LABELS}
    last_save_time = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            continue
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        frame_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        h, w = frame_bgr.shape[:2]
        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            handedness = results.multi_handedness[0].classification[0].label
            if handedness == "Right":
                handedness = "Left"
            elif handedness == "Left":
                handedness = "Right"
            mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            coords = []
            for lm in hand_landmarks.landmark:
                coords.extend([lm.x, lm.y, lm.z])
            mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            coords = []
            for lm in hand_landmarks.landmark:
                coords.extend([lm.x, lm.y, lm.z])

            # UI info
            cv2.putText(frame_bgr, f"Label: {GESTURE_LABELS[current_label_idx] if current_label_idx is not None else '-'}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            cv2.putText(frame_bgr, f"Saved: {saved_counts}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

        else:
            coords = None
            handedness = None
            cv2.putText(frame_bgr, "Nema ruke", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

        cv2.imshow("Prikupljanje podataka", frame_bgr)
        key = cv2.waitKey(1) & 0xFF

        # Odabir labele tipkama 0..9.. itd.
        if ord('0') <= key <= ord('9'):
            idx = key - ord('0')
            if idx < len(GESTURE_LABELS):
                current_label_idx = idx
        elif key == ord('q'):
            break
        elif key == ord('s') and coords is not None and current_label_idx is not None:
            now = time.time()
            if now - last_save_time < SAVE_PAUSE_SEC:
                continue
            last_save_time = now
            label = GESTURE_LABELS[current_label_idx]
            sample = {
                "label": label,
                "handedness": handedness,
                "landmarks": coords,
            }
            fname = os.path.join(RAW_DIR, f"{int(now*1000)}_{uuid.uuid4().hex}.json")
            with open(fname, "w") as f:
                json.dump(sample, f)
            saved_counts[label] += 1

cap.release()
cv2.destroyAllWindows()
print("Gotovo prikupljanje.")


Oznake: ['OK', 'L', 'Ljubav', 'Rock!', 'Thumbs Up', 'Thumbs Down', 'Peace', 'Point', 'Fist', 'Open Hand', 'Poziv', 'Finger gun']
Pritisni broj 0-11 za labelu, 's' za spremanje uzorka, 'q' za izlaz


I0000 00:00:1757529613.552145   58558 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1
W0000 00:00:1757529613.576758  571406 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757529613.595873  571408 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Gotovo prikupljanje.


In [88]:
# Treniranje TensorFlow modela s NORMALIZACIJOM (koristi samo x,y koordinate: 21*2 = 42)
import glob

files = sorted(glob.glob(os.path.join(RAW_DIR, "*.json")))
print(f"Datoteka: {len(files)}")
if len(files) == 0:
    print("Nema prikupljenih podataka. Pokreni celiju za prikupljanje.")
else:
    X = []
    y = []
    for fp in files:
        with open(fp, "r") as f:
            sample = json.load(f)
        raw_label = sample["label"]
        mapped_label = TF_LABEL_REMAP.get(raw_label, raw_label)
        if mapped_label not in TF_LABELS:
            # Preskoci uzorke cije mape nema u TF_LABELS
            continue
        coords = sample["landmarks"]  # [x,y,z]*21
        handedness = sample.get("handedness")
        
        # Koristi normalizaciju umjesto sirovih koordinata
        xy = preprocess_landmarks_xy(
            [type("L",(object,),{"x":coords[3*i+0],"y":coords[3*i+1]}) for i in range(21)],
            handedness=handedness
        )
        X.append(xy)
        y.append(TF_LABELS.index(mapped_label))

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    print("X shape:", X.shape, "y shape:", y.shape)

    if len(X) == 0:
        print("Nema valjanih uzoraka nakon remapiranja. Provjeri TF_LABELS/TF_LABEL_REMAP.")
    else:
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y if len(np.unique(y))>1 else None
        )

        # Poboljšani model s normalizacijom
        model_tf = tf.keras.models.Sequential([
            tf.keras.layers.Input((21 * 2, )),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dense(128, activation='relu'),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
        ])

        model_tf.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy"),
        ]

        history = model_tf.fit(
            X_train, y_train,
            validation_data=(X_val, y_val) if len(X_val) else None,
            epochs=100,
            batch_size=32,
            callbacks=callbacks,
            verbose=1
        )

        model_tf.save(TF_MODEL_PATH)
        with open(META_PATH, "w") as f:
            json.dump({"labels": TF_LABELS}, f)
        print("TF model spremljen u:", TF_MODEL_PATH)


Datoteka: 1067
X shape: (1067, 42) y shape: (1067,)
Epoch 1/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.2016 - loss: 2.5724 - val_accuracy: 0.3271 - val_loss: 1.7424
Epoch 2/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4818 - loss: 1.6724 - val_accuracy: 0.5187 - val_loss: 1.3972
Epoch 3/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5826 - loss: 1.2879 - val_accuracy: 0.6589 - val_loss: 1.1009
Epoch 4/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6413 - loss: 1.0852 - val_accuracy: 0.7336 - val_loss: 0.8854
Epoch 5/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.6776 - loss: 0.9784 - val_accuracy: 0.8224 - val_loss: 0.7172
Epoch 6/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6753 - loss: 0.9139 - val_accuracy: 0.8551 - val_loss: 0.6154
Epoch 7/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7093 - loss: 0.8449 - val_accuracy: 0.8551 - val_loss: 0.5433
Epoch 8/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step 

In [89]:
# Ucitavanje spremljenog TF modela i mapa labela
model_tf = None
LABELS = GESTURE_LABELS

if os.path.exists(META_PATH):
    with open(META_PATH, "r") as f:
        meta = json.load(f)
    LABELS = meta.get("labels", GESTURE_LABELS)

if os.path.exists(TF_MODEL_PATH):
    try:
        model_tf = tf.keras.models.load_model(TF_MODEL_PATH)
        print("Ucitani TF model.")
    except Exception as e:
        print("Greska pri ucitavanju TF modela:", e)
else:
    print("TF model nije pronadjen. Pokreni TF trening celiju.")


Ucitani TF model.


In [91]:
# Real-time inferencija s NORMALIZACIJOM (samo TF model)
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera nije dostupna")

with mp_hands.Hands(static_image_mode=False,
                    max_num_hands=1,
                    min_detection_confidence=0.7,
                    min_tracking_confidence=0.5) as hands:
    while True:
        ok, frame = cap.read()
        if not ok:
            continue
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        frame_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        pred_label = None
        pred_prob = None

        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            # TF features: normalizirani x,y (42)
            handed = None
            if results.multi_handedness:
                handed = results.multi_handedness[0].classification[0].label
            
            feats_xy = preprocess_landmarks_xy(hand_landmarks.landmark, handedness=handed)
            feats_xy = feats_xy.reshape(1, -1)

            if 'model_tf' in globals() and model_tf is not None:
                probs = model_tf.predict(feats_xy, verbose=0)[0]
                idx = int(np.argmax(probs))
                pred_label = LABELS[idx]
                pred_prob = float(np.max(probs))

        # UI
        cv2.putText(frame_bgr, f"Model: {'TF' if ('model_tf' in globals() and model_tf is not None) else 'Nije ucitan'}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
        if pred_label is not None and pred_prob >= 0.6:  # prag pouzdanosti
            cv2.putText(frame_bgr, f"Predikcija: {pred_label} ({pred_prob:.2f})", (10, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        elif pred_label is not None:
            cv2.putText(frame_bgr, f"Nepouzdano: {pred_label} ({pred_prob:.2f})", (10, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

        cv2.imshow("Inferencija (TF)", frame_bgr)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


I0000 00:00:1757530148.029533   58558 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1
W0000 00:00:1757530148.064008  608531 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757530148.083471  608536 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [1]:
# Provjera koliko imaš podataka po gesti
import glob, json, collections, os

cnt = collections.Counter()
for fp in glob.glob(os.path.join(RAW_DIR, "*.json")):
    with open(fp) as f:
        cnt[json.load(f)["label"]] += 1

for label, n in sorted(cnt.items(), key=lambda x: x[0]):
    print(f"{label}: {n}")
print(f"Ukupno: {sum(cnt.values())}")


NameError: name 'RAW_DIR' is not defined

In [16]:
%pip uninstall -y ml-dtypes jax jaxlib
%pip install "ml_dtypes>=0.3.1,<0.4"
# (po želji) osiguraj TF za Apple Silicon:
%pip install -U tensorflow-macos tensorflow-metal
!pip install -U tensorflow-macos tensorflow-metal
!pip install -q tensorflow-macos tensorflow-metal || pip install -q tensorflow

Found existing installation: ml-dtypes 0.3.2
Uninstalling ml-dtypes-0.3.2:
  Successfully uninstalled ml-dtypes-0.3.2
Note: you may need to restart the kernel to use updated packages.
  Using cached ml_dtypes-0.3.2-cp311-cp311-macosx_10_9_universal2.whl.metadata (20 kB)
Using cached ml_dtypes-0.3.2-cp311-cp311-macosx_10_9_universal2.whl (389 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [17]:
!pip install mediapipe opencv-python

  Using cached ml_dtypes-0.5.3-cp311-cp311-macosx_10_9_universal2.whl.metadata (8.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 MB 1.6 MB/s eta 0:00:0000:0100:01
Using cached ml_dtypes-0.5.3-cp311-cp311-macosx_10_9_universal2.whl (667 kB)
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [jax]2/3 [jax]ib]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.2 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.3 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [18]:
import mediapipe as mp
import cv2
import numpy as np
import uuid
import os

In [19]:
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands


In [3]:
import cv2
import mediapipe as mp

cap = cv2.VideoCapture(0)

with mp.solutions.hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        # Konverzija u RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        

        # Postavljanje image-a kao ne-editable
        image.flags.writeable = False
        results = hands.process(image)
        # Detekcija ruku
        image.flags.writeable = True
        # Konverzija u BGR
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        

        # Ispisivanje/Renderanje ruku
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS,
                                          mp_drawing.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=3),
                                          mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=2))

        cv2.imshow('Pracenje ruke', frame)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [7]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

def prepoznaj_gestu(landmarks, handedness):
    def is_extended(tip, pip): return tip[1] < pip[1] - 0.04  # Povećana preciznost
    def is_curled(tip, pip): return tip[1] > pip[1] + 0.03   # Povećana preciznost

    thumb_tip, thumb_ip = landmarks[4], landmarks[3]
    index_tip, index_pip = landmarks[8], landmarks[6]
    middle_tip, middle_pip = landmarks[12], landmarks[10]
    ring_tip, ring_pip = landmarks[16], landmarks[14]
    pinky_tip, pinky_pip = landmarks[20], landmarks[18]

    # Prilagodba thumb detekcije za lijevu i desnu ruku
    if handedness == "Right":
        # Za desnu ruku, thumb je extended ako je lijevo od MCP joint-a
        thumb_extended = thumb_tip[0] < landmarks[2][0] - 0.02  # Povećana preciznost
        thumb_down = thumb_tip[1] > thumb_ip[1] + 0.02
        thumb_up = thumb_tip[1] < thumb_ip[1] - 0.02
    else:
        # Za lijevu ruku, thumb je extended ako je desno od MCP joint-a
        thumb_extended = thumb_tip[0] > landmarks[2][0] + 0.02  # Povećana preciznost
        thumb_down = thumb_tip[1] > thumb_ip[1] + 0.02
        thumb_up = thumb_tip[1] < thumb_ip[1] - 0.02

    index_extended = is_extended(index_tip, index_pip)
    middle_extended = is_extended(middle_tip, middle_pip)
    ring_extended = is_extended(ring_tip, ring_pip)
    pinky_extended = is_extended(pinky_tip, pinky_pip)

    index_curled = is_curled(index_tip, index_pip)
    middle_curled = is_curled(middle_tip, middle_pip)
    ring_curled = is_curled(ring_tip, ring_pip)
    pinky_curled = is_curled(pinky_tip, pinky_pip)
    

    # OK - prilagodba za obje ruke
    if handedness == "Right":
        # Za desnu ruku, thumb i index se dodiruju
        ok_condition = abs(thumb_tip[0] - index_tip[0]) < 0.03 and abs(thumb_tip[1] - index_tip[1]) < 0.03
    else:
        # Za lijevu ruku, thumb i index se dodiruju
        ok_condition = abs(thumb_tip[0] - index_tip[0]) < 0.03 and abs(thumb_tip[1] - index_tip[1]) < 0.03
    
    if ok_condition:
        return "OK", "Sve je u redu!"

    # L 
    if thumb_extended and index_extended and middle_curled and ring_curled and pinky_curled:
        return "L", "L!"

    # Ljubav 
    if thumb_extended and index_extended and middle_curled and ring_curled and pinky_extended:
        return "Ljubav", "Volim te! ❤️"

    # Rock! 
    if not thumb_extended and index_extended and middle_curled and ring_curled and pinky_extended:
        return "Rock!", "Rock on! 🤘"

    # Thumbs Up
    if thumb_up and index_curled and middle_curled and ring_curled and pinky_curled:
        return "Thumbs Up", "Pozitivno!"

    # Thumbs Down
    if thumb_down and index_curled and middle_curled and ring_curled and pinky_curled:
        return "Thumbs Down", "Negativno!"

    # Peace
    if index_extended and middle_extended and ring_curled and pinky_curled and not thumb_extended:
        return "Peace", "Mir!"

    # Point
    if index_extended and middle_curled and ring_curled and pinky_curled:
        return "Point", "Pogledaj ovdje!"

    # Fist
    if index_curled and middle_curled and ring_curled and pinky_curled:
        return "Fist", "Pesnica!"

    # Open Hand
    if index_extended and middle_extended and ring_extended and pinky_extended and thumb_extended:
        return "Open Hand", "Dobrodosao!"

    # Poziv
    if pinky_extended and thumb_extended and index_curled and middle_curled and ring_curled:
        return "Poziv", "Poziv!"

    # Finger gun
    if index_extended and middle_extended and thumb_extended and pinky_curled and ring_curled:
        return "Finger gun", "Pistolj"
    

    return "Nepoznato", "Nepoznata gesta."

def pracenje_gesti():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Greska: Kamera nije dostupna.")
        return

    with mp_hands.Hands(static_image_mode=False, max_num_hands=2, min_detection_confidence=0.8, min_tracking_confidence=0.5) as hands:
        while True:
            ret, frame = cap.read()
            if not ret:
                continue

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb_frame)

            # Crni okvir za tekst
            cv2.rectangle(frame, (10, 10), (400, 80), (0, 0, 0), -1)

            if results.multi_hand_landmarks:
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    label = handedness.classification[0].label #lijeva ili desna ruka
                    mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                    landmarks = [[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]
                    
                    
                    gesta, poruka = prepoznaj_gestu(landmarks, label)

                    # Tekst unutar okvira
                    cv2.putText(frame, f"Gesta: {gesta}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                    cv2.putText(frame, f"Poruka: {poruka}", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
                    cv2.putText(frame, f"Ruka: {label}", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)

                    # Vizualni simboli
                    if gesta == "OK":
                        cv2.circle(frame, (50, 150), 30, (0, 255, 0), -1)
                    elif gesta == "Thumbs Up":
                        cv2.rectangle(frame, (50, 150), (80, 200), (0, 255, 0), -1)
                    elif gesta == "Thumbs Down":
                        cv2.rectangle(frame, (50, 200), (80, 150), (0, 0, 255), -1)
                    elif gesta == "Peace":
                        cv2.putText(frame, "✌", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
                    elif gesta == "Point":
                        cv2.line(frame, (50, 150), (100, 150), (255, 0, 0), 3)
                    elif gesta == "Fist":
                        cv2.circle(frame, (75, 175), 25, (128, 128, 128), -1)
                    elif gesta == "Open Hand":
                        cv2.rectangle(frame, (50, 150), (100, 200), (255, 255, 0), -1)
                    elif gesta == "L":
                        cv2.putText(frame, "L", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
                    elif gesta == "Ljubav":
                        cv2.putText(frame, "❤", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                    elif gesta == "Rock!":
                        cv2.putText(frame, "🤘", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 255), 2)
                    elif gesta == "Poziv":
                        cv2.putText(frame, "Poziv", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 255), 2)
                    elif gesta == "Finger gun":
                        cv2.putText(frame, "Pistolj", (50, 180), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 255), 2)
            else:
                cv2.putText(frame, "Nema ruke u vidnom polju", (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

            cv2.imshow("Prepoznavanje gesti ruke", frame)
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

# Pokretanje
pracenje_gesti()

In [20]:
pip install -q scikit-learn joblib matplotlib seaborn



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import time
import numpy as np
from sklearn.model_selection import train_test_split

# TensorFlow (za Keras model)
import tensorflow as tf

# Sirove labele koje koristi prikupljanje (tipke 0..)
GESTURE_LABELS = [
    "OK",
    "L",
    "Ljubav",
    "Rock!",
    "Thumbs Up",
    "Thumbs Down",
    "Peace",
    "Point",
    "Fist",
    "Open Hand",
    "Poziv",
    "Finger gun",
]

# TF labele za treniranje/inferenciju (ovdje promijeni nazive/redoslijed)
TF_LABELS = [
    "OK",
    "L",
    "Ljubav",
    "Rock!",
    "Thumbs Up",
    "Thumbs Down",
    "Peace",
    "Point",
    "Fist",
    "Open Hand",
    "Poziv",
    "Finger gun",
]

NUM_CLASSES = len(TF_LABELS)

DATA_DIR = "data"
RAW_DIR = os.path.join(DATA_DIR, "raw")
MODEL_DIR = "models"
TF_MODEL_PATH = os.path.join(MODEL_DIR, "hand_gesture_tf.keras")
META_PATH = os.path.join(MODEL_DIR, "labels.json")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Spremno. Dirs:", RAW_DIR, MODEL_DIR)


Spremno. Dirs: data/raw models


In [ ]:
# Prikupljanje podataka: spremi 21*3 = 63 vrijednosti po ruci + handedness
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

SAMPLES_PER_CLASS = 80  # promijeni po potrebi
SAVE_PAUSE_SEC = 0.1

print("Oznake:", GESTURE_LABELS)
print("Pritisni broj 0-{} za labelu, 's' za spremanje uzorka, 'q' za izlaz".format(len(GESTURE_LABELS)-1))

current_label_idx = None

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera nije dostupna")

with mp_hands.Hands(static_image_mode=False,
                    max_num_hands=1,
                    min_detection_confidence=0.7,
                    min_tracking_confidence=0.5) as hands:
    saved_counts = {label: 0 for label in GESTURE_LABELS}
    last_save_time = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            continue
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        frame_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        h, w = frame_bgr.shape[:2]
        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            handedness = results.multi_handedness[0].classification[0].label
            if handedness == "Right":
                handedness = "Left"
            elif handedness == "Left":
                handedness = "Right"
            mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            coords = []
            for lm in hand_landmarks.landmark:
                coords.extend([lm.x, lm.y, lm.z])
            mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            coords = []
            for lm in hand_landmarks.landmark:
                coords.extend([lm.x, lm.y, lm.z])

            # UI info
            cv2.putText(frame_bgr, f"Label: {GESTURE_LABELS[current_label_idx] if current_label_idx is not None else '-'}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            cv2.putText(frame_bgr, f"Saved: {saved_counts}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

        else:
            coords = None
            handedness = None
            cv2.putText(frame_bgr, "Nema ruke", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

        cv2.imshow("Prikupljanje podataka", frame_bgr)
        key = cv2.waitKey(1) & 0xFF

        # Odabir labele tipkama 0..9.. itd.
        if ord('0') <= key <= ord('9'):
            idx = key - ord('0')
            if idx < len(GESTURE_LABELS):
                current_label_idx = idx
        elif key == ord('q'):
            break
        elif key == ord('s') and coords is not None and current_label_idx is not None:
            now = time.time()
            if now - last_save_time < SAVE_PAUSE_SEC:
                continue
            last_save_time = now
            label = GESTURE_LABELS[current_label_idx]
            sample = {
                "label": label,
                "handedness": handedness,
                "landmarks": coords,
            }
            fname = os.path.join(RAW_DIR, f"{int(now*1000)}_{uuid.uuid4().hex}.json")
            with open(fname, "w") as f:
                json.dump(sample, f)
            saved_counts[label] += 1

cap.release()
cv2.destroyAllWindows()
print("Gotovo prikupljanje.")


Oznake: ['OK', 'L', 'Ljubav', 'Rock!', 'Thumbs Up', 'Thumbs Down', 'Peace', 'Point', 'Fist', 'Open Hand', 'Poziv', 'Finger gun']
Pritisni broj 0-11 za labelu, 's' za spremanje uzorka, 'q' za izlaz


I0000 00:00:1757249768.723312  111986 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1
W0000 00:00:1757249768.747251  132899 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757249768.762626  132903 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Gotovo prikupljanje.


In [71]:
# Treniranje TensorFlow modela (koristi samo x,y koordinate: 21*2 = 42)
import glob

files = sorted(glob.glob(os.path.join(RAW_DIR, "*.json")))
print(f"Datoteka: {len(files)}")
if len(files) == 0:
    print("Nema prikupljenih podataka. Pokreni celiju za prikupljanje.")
else:
    X = []
    y = []
    for fp in files:
        with open(fp, "r") as f:
            sample = json.load(f)
        raw_label = sample["label"]
        mapped_label = TF_LABEL_REMAP.get(raw_label, raw_label)
        if mapped_label not in TF_LABELS:
            # Preskoci uzorke cije mape nema u TF_LABELS
            continue
        coords = sample["landmarks"]  # [x,y,z]*21
        # Uzmi samo x,y -> 42 vrijednosti
        xy = []
        for i in range(21):
            xy.append(coords[3*i + 0])
            xy.append(coords[3*i + 1])
        X.append(xy)
        y.append(TF_LABELS.index(mapped_label))

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    print("X shape:", X.shape, "y shape:", y.shape)

    if len(X) == 0:
        print("Nema valjanih uzoraka nakon remapiranja. Provjeri TF_LABELS/TF_LABEL_REMAP.")
    else:
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y if len(np.unique(y))>1 else None
        )

        # Definirani model
        model_tf = tf.keras.models.Sequential([
            tf.keras.layers.Input((21 * 2, )),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(20, activation='relu'),
            tf.keras.layers.Dropout(0.4),
            tf.keras.layers.Dense(10, activation='relu'),
            tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
        ])

        model_tf.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_accuracy"),
        ]

        history = model_tf.fit(
            X_train, y_train,
            validation_data=(X_val, y_val) if len(X_val) else None,
            epochs=100,
            batch_size=32,
            callbacks=callbacks,
            verbose=1
        )

        model_tf.save(TF_MODEL_PATH)
        with open(META_PATH, "w") as f:
            json.dump({"labels": TF_LABELS}, f)
        print("TF model spremljen u:", TF_MODEL_PATH)


Datoteka: 817
X shape: (817, 42) y shape: (817,)
Epoch 1/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.0597 - loss: 3.0180 - val_accuracy: 0.1098 - val_loss: 2.7249
Epoch 2/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0689 - loss: 2.8147 - val_accuracy: 0.1159 - val_loss: 2.5990
Epoch 3/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.1087 - loss: 2.7154 - val_accuracy: 0.1098 - val_loss: 2.5249
Epoch 4/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0919 - loss: 2.6540 - val_accuracy: 0.1159 - val_loss: 2.4631
Epoch 5/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1179 - loss: 2.6137 - val_accuracy: 0.0976 - val_loss: 2.4129
Epoch 6/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1057 - loss: 2.5857 - val_accuracy: 0.0976 - val_loss: 2.3806
Epoch 7/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1072 - loss: 2.4996 - val_accuracy: 0.1037 - val_loss: 2.3635
Epoch 8/100
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - a

In [72]:
# Ucitavanje spremljenog TF modela i mapa labela
model_tf = None
LABELS = GESTURE_LABELS

if os.path.exists(META_PATH):
    with open(META_PATH, "r") as f:
        meta = json.load(f)
    LABELS = meta.get("labels", GESTURE_LABELS)

if os.path.exists(TF_MODEL_PATH):
    try:
        model_tf = tf.keras.models.load_model(TF_MODEL_PATH)
        print("Ucitani TF model.")
    except Exception as e:
        print("Greska pri ucitavanju TF modela:", e)
else:
    print("TF model nije pronadjen. Pokreni TF trening celiju.")


Ucitani TF model.


In [73]:
# Real-time inferencija (samo TF model)
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Kamera nije dostupna")

with mp_hands.Hands(static_image_mode=False,
                    max_num_hands=1,
                    min_detection_confidence=0.7,
                    min_tracking_confidence=0.5) as hands:
    while True:
        ok, frame = cap.read()
        if not ok:
            continue
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = hands.process(image)
        image.flags.writeable = True
        frame_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        pred_label = None
        pred_prob = None

        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            mp_drawing.draw_landmarks(frame_bgr, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            # TF features: samo x,y (42)
            feats_xy = []
            for lm in hand_landmarks.landmark:
                feats_xy.extend([lm.x, lm.y])
            feats_xy = np.array(feats_xy, dtype=np.float32).reshape(1, -1)

            if 'model_tf' in globals() and model_tf is not None:
                probs = model_tf.predict(feats_xy, verbose=0)[0]
                idx = int(np.argmax(probs))
                pred_label = LABELS[idx]
                pred_prob = float(np.max(probs))

        # UI
        cv2.putText(frame_bgr, f"Model: {'TF' if ('model_tf' in globals() and model_tf is not None) else 'Nije ucitan'}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)
        if pred_label is not None:
            cv2.putText(frame_bgr, f"Predikcija: {pred_label} ({pred_prob:.2f})", (10, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        cv2.imshow("Inferencija (TF)", frame_bgr)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()


I0000 00:00:1757527956.092018   58558 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1
W0000 00:00:1757527956.109539  442923 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1757527956.116626  442923 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [30]:
# Brza provjera: predikcija nad jednim uzorkom (samo TF)
import glob

files = sorted(glob.glob(os.path.join(RAW_DIR, "*.json")))
if not files:
    print("Nema podataka za provjeru.")
else:
    with open(files[0], "r") as f:
        s = json.load(f)

    # TF features (x,y)
    coords = s["landmarks"]
    xy = []
    for i in range(21):
        xy.append(coords[3*i + 0])
        xy.append(coords[3*i + 1])
    x_tf = np.array(xy, dtype=np.float32).reshape(1, -1)

    if 'model_tf' in globals() and model_tf is not None:
        probs = model_tf.predict(x_tf, verbose=0)[0]
        idx = int(np.argmax(probs))
        print("TF predikcija:", LABELS[idx], float(np.max(probs)))
    else:
        print("TF model nije ucitan.")


TF predikcija: OK 0.12329189479351044
